# Kalshi favourite-longshot bias: replication, extension and favourite buying

Module 3, hedge fund strategies. Plan: [PLAN.md](PLAN.md).

1. Replicate Bürgi, Deng and Whelan, *Makers and Takers: The Economics of the Kalshi Prediction Market*, on their sample (2021 to April 2025).
2. Extend the same tables out of sample (May 2025 to October 2026).
3. Backtest a weekly favourite-buying portfolio (85–97c), then improve it with a calibration model.

This notebook does the data cleaning and evaluation. The data comes from `download_data.py`, which saves it under `data/`:

```
.venv/bin/python download_data.py
```

The notebook never calls the API, so it runs top to bottom in seconds once the data is on disk.

## 1. Setup

In [1]:
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [2]:
DATA = Path("data")
INCLUDE_PAPER_EXTRA = True    # add sports and daily crypto/index markets closing before May 2025 (download_data.py paper-extra)

# --- Sample dates ---
PAPER_START = dt.datetime(2021, 1, 1, tzinfo=dt.timezone.utc)
PAPER_END   = dt.datetime(2025, 5, 1, tzinfo=dt.timezone.utc)    # paper sample ends April 2025
OOS_END     = dt.datetime(2026, 11, 1, tzinfo=dt.timezone.utc)   # out of sample: May 2025 to 31 Oct 2026

# --- Paper sample filters (section 4) ---
MIN_VOLUME_DOLLARS = 1_000
MAX_FINAL_SPREAD = 0.20
MIN_HOURS_OPEN = 24

# --- Strategy (section 7) ---
TAKER_FEE_RATE = 0.07      # fee = 0.07 * p * (1 - p) per contract, rounded up to the cent
INTEREST_RATE = 0.035      # annual interest on cash and positions
FAVOURITE_MIN, FAVOURITE_MAX = 0.85, 0.97
MIN_DAYS_TO_CLOSE, MAX_DAYS_TO_CLOSE = 1, 21
EVENT_CAP = 0.03

## 2. Load data

Three files written by `download_data.py`. Values are stored as text, exactly as the API returned them; section 4 converts them.

- `series.parquet`: every Kalshi series with its category and frequency.
- `markets.parquet`: every settled market in the series we keep (sports, hourly, 15-minute and combo series are left out).
- `candles.parquet`: daily price, bid, ask and volume for markets with at least 1,000 contracts traded that were open at least 24 hours.

The main download skips sports and every series Kalshi tags as hourly or 15-minute. The paper keeps any market open at least 24 hours, which includes sports from January 2025 and the daily markets inside "hourly" crypto and index series (for example KXBTCD and INXU). For the paper period only, `download_data.py paper-extra` adds those markets with their candles and trades under `data/paper_extra/`. Section 4's 24-hour filter keeps the truly hourly markets out.

In [3]:
series = pl.read_parquet(DATA / "series.parquet")
markets = pl.read_parquet(DATA / "markets.parquet")
candles = pl.read_parquet(DATA / "candles.parquet")

extra_tickers = pl.Series("ticker", [], dtype=pl.String)
if INCLUDE_PAPER_EXTRA:
    extra_markets = pl.read_parquet(DATA / "paper_extra" / "markets.parquet")
    extra_tickers = extra_markets["ticker"]
    markets = pl.concat([markets, extra_markets], how="diagonal_relaxed")
    candles = pl.concat([candles, pl.read_parquet(DATA / "paper_extra" / "candles.parquet")], how="diagonal_relaxed")

print(f"{len(series):,} series, {len(markets):,} settled markets, "
      f"{len(candles):,} daily candles for {candles['ticker'].n_unique():,} markets")

16,191 series, 836,960 settled markets, 3,700,181 daily candles for 241,635 markets


Markets with candles, by category and the year they closed.

In [4]:
with_candles = (
    markets
    .filter(pl.col("ticker").is_in(candles["ticker"].unique()))
    .join(series.select(pl.col("ticker").alias("series_ticker"), "category"), on="series_ticker", how="left")
    .with_columns(pl.col("close_time").str.to_datetime(time_zone="UTC").dt.year().alias("year"))
)
(with_candles
 .group_by("category", "year").len()
 .pivot(on="year", index="category", values="len", sort_columns=True)
 .sort("category"))

/var/folders/wm/nxkcxnfx12v9kprllw20m8qh0000gn/T/ipykernel_3202/3187894488.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ticker").is_in(candles["ticker"].unique()))


category,2021,2022,2023,2024,2025,2026
str,u32,u32,u32,u32,u32,u32
null,null,null,1171,110,null,null
"""Climate and Weather""",518,2524,2026,5616,13894,54363
"""Commodities""",null,146,123,71,44,15188
"""Companies""",null,null,6,10,39,81
"""Crypto""",null,null,null,5145,3397,1624
…,…,…,…,…,…,…
"""Science and Technology""",13,8,1,103,390,4159
"""Social""",null,null,null,null,38,9
"""Sports""",null,null,null,1,1745,null


## 3. Helper functions

Fee, settlement portfolio, calibration table and performance statistics. To be written with sections 5 and 7.

In [5]:
def taker_fee(price, contracts=1):
    """Kalshi taker fee in dollars: 0.07 * contracts * p * (1 - p), rounded up to the cent."""
    return np.ceil(TAKER_FEE_RATE * contracts * price * (1 - price) * 100 - 1e-9) / 100

## 4. Paper sample

The paper's filters (section 3.1 of Bürgi, Deng and Whelan):

1. Markets from Kalshi's launch in 2021 through April 2025 (we also keep May 2025 to October 2026 for section 6).
2. Total trading volume at close of at least $1,000. Kalshi reports volume in contracts, each with $1 notional, so this is `volume_fp >= 1,000`.
3. Final bid–ask spread of 20c or less.
4. Open at least 24 hours, which drops the hourly crypto and index markets.

The final spread comes from the last daily candle that ends no later than one hour after the market closes. Later candles show the book after trading stopped, with a bid of 0 and an ask of 1.

In [6]:
categories = series.select(pl.col("ticker").alias("series_ticker"), "category")

markets_clean = (
    markets
    .select("ticker", "event_ticker", "series_ticker", "title", "yes_sub_title", "market_type", "result", "source",
            pl.col("volume_fp").cast(pl.Float64).alias("volume"),
            pl.col("open_time").str.to_datetime(time_zone="UTC").alias("open"),
            pl.col("close_time").str.to_datetime(time_zone="UTC").alias("close"))
    .join(categories, on="series_ticker", how="left")
)

final_quote = (
    candles
    .select("ticker",
            pl.from_epoch(pl.col("end_period_ts").cast(pl.Int64), time_unit="s").dt.replace_time_zone("UTC").alias("candle_end"),
            pl.col("yes_bid_close").cast(pl.Float64).alias("final_bid"),
            pl.col("yes_ask_close").cast(pl.Float64).alias("final_ask"))
    .join(markets_clean.select("ticker", "close"), on="ticker")
    .filter(pl.col("candle_end") <= pl.col("close") + pl.duration(hours=1))
    .sort("candle_end")
    .group_by("ticker").last()
    .select("ticker", "final_bid", "final_ask", (pl.col("final_ask") - pl.col("final_bid")).alias("final_spread"))
)

In [7]:
sample = (
    markets_clean
    .join(final_quote, on="ticker", how="inner")
    .filter(
        (pl.col("market_type") == "binary")
        & pl.col("result").is_in(["yes", "no"])
        & (pl.col("volume") >= MIN_VOLUME_DOLLARS)
        & (pl.col("final_spread") <= MAX_FINAL_SPREAD)
        & ((pl.col("close") - pl.col("open")) >= pl.duration(hours=MIN_HOURS_OPEN))
        & (pl.col("close") >= PAPER_START) & (pl.col("close") < OOS_END)
    )
    .with_columns(pl.when(pl.col("close") < PAPER_END).then(pl.lit("paper")).otherwise(pl.lit("oos")).alias("period"))
)

sample.group_by("period").agg(
    pl.len().alias("yes_contracts"),
    pl.col("event_ticker").n_unique().alias("events"),
).sort("period", descending=True)

period,yes_contracts,events
str,u32,u32
"""paper""",41739,11678
"""oos""",156916,24732


The paper reports **46,282 Yes contracts and 12,403 events** for 2021 to April 2025. Our counts are lower mainly because sports is left out; the paper notes its results do not change without the sports category.

How many markets each filter removes, for the paper period:

In [8]:
paper_period = markets_clean.filter((pl.col("close") >= PAPER_START) & (pl.col("close") < PAPER_END)).join(final_quote, on="ticker", how="left")
steps = [
    ("settled Yes or No, binary", (pl.col("market_type") == "binary") & pl.col("result").is_in(["yes", "no"])),
    ("volume >= $1,000", pl.col("volume") >= MIN_VOLUME_DOLLARS),
    ("open >= 24 hours", (pl.col("close") - pl.col("open")) >= pl.duration(hours=MIN_HOURS_OPEN)),
    ("final spread <= 20c", pl.col("final_spread") <= MAX_FINAL_SPREAD),
]
rows, keep = [("all settled markets", len(paper_period))], paper_period
for name, condition in steps:
    keep = keep.filter(condition)
    rows.append((name, len(keep)))
pl.DataFrame(rows, schema=["after filter", "markets"], orient="row")

after filter,markets
str,i64
"""all settled markets""",306160
"""settled Yes or No, binary""",306105
"""volume >= $1,000""",79813
"""open >= 24 hours""",53044
"""final spread <= 20c""",41739


Save the sample's tickers so `download_data.py trades` can fetch their trades.

In [9]:
sample.select("ticker", "period").write_parquet(DATA / "trade_tickers.parquet")
print(f"Saved {len(sample):,} tickers to data/trade_tickers.parquet")

Saved 198,655 tickers to data/trade_tickers.parquet


### Price snapshots

The paper takes the last trade before the market closed, then goes back in 24-hour steps to ten days before close. Its footnote 3: if the last trade was at 5:57pm on the closing day, it also records the last trade before 5:57pm on each earlier day that had trades. So each Yes contract has at most 11 prices.

Here the snapshot `k` days before close is the price as of exactly `k` × 24 hours before the last trade: the most recent trade at or before that moment, even if it happened on an earlier day. A contract gets a snapshot for day `k` if it had traded by then. We tested four other readings (24-hour windows, calendar days, same clock time on each calendar day, and windows counted from the market's close time). This one reproduces the paper's Table 1 closely and the others do not.

When one order fills at several prices, all the fills share a timestamp. The order book is swept from the best price outwards, so the last fill is the highest Yes price when the taker bought Yes and the lowest when the taker bought No. Ordering ties this way makes our last trade match Kalshi's reported last price.

Trades are one file per series (`download_data.py trades`). The paper-period files are small enough to load in full.

In [10]:
trade_files = [f for f in sorted((DATA / "trades").glob("*__paper.parquet")) if pl.read_parquet_schema(f)]
if INCLUDE_PAPER_EXTRA:
    trade_files.append(DATA / "paper_extra" / "trades.parquet")
trades = (
    pl.concat([pl.read_parquet(f) for f in trade_files], how="diagonal_relaxed")
    .select("ticker",
            pl.col("created_time").str.to_datetime(time_zone="UTC", format="%Y-%m-%dT%H:%M:%S%.fZ").alias("time"),
            pl.col("yes_price_dollars").cast(pl.Float64).alias("yes_price"),
            pl.col("count_fp").cast(pl.Float64).alias("contracts"),
            pl.col("taker_outcome_side").alias("taker_side"))
)
print(f"{len(trades):,} trades for {trades['ticker'].n_unique():,} markets")

14,164,848 trades for 187,864 markets


In [11]:
last_trade_time = trades.group_by("ticker").agg(pl.col("time").max().alias("last_time"))
sweep_order = pl.when(pl.col("taker_side") == "yes").then(pl.col("yes_price")).otherwise(-pl.col("yes_price"))

snapshot_times = (
    last_trade_time
    .join(pl.DataFrame({"days_before": list(range(11))}, schema={"days_before": pl.Int32}), how="cross")
    .with_columns((pl.col("last_time") - pl.duration(days=1) * pl.col("days_before")).alias("snapshot_time"))
    .sort("snapshot_time")
)
trades_in_order = trades.sort(pl.col("time"), sweep_order).select("ticker", "time", "yes_price")

snapshots = (
    snapshot_times
    .join_asof(trades_in_order, left_on="snapshot_time", right_on="time", by="ticker", strategy="backward")
    .filter(pl.col("time").is_not_null())                      # the contract had not traded yet
    .select("ticker", "days_before", "time", "yes_price")
    .join(sample.select("ticker", "event_ticker", "category", "result", "close"), on="ticker")
    .sort("ticker", "days_before")
)

/var/folders/wm/nxkcxnfx12v9kprllw20m8qh0000gn/T/ipykernel_3202/3408946329.py:14: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(trades_in_order, left_on="snapshot_time", right_on="time", by="ticker", strategy="backward")


The paper also dropped 63 contracts whose last trade did not match the final price Kalshi reports for the market (footnote 2). We do the same check.

In [12]:
final_price = markets.select("ticker", pl.col("last_price_dollars").cast(pl.Float64).alias("reported_last_price"))
mismatch = (
    snapshots.filter(pl.col("days_before") == 0)
    .join(final_price, on="ticker")
    .filter((pl.col("yes_price") - pl.col("reported_last_price")).abs() > 1e-6)
    ["ticker"]
)
snapshots = snapshots.filter(~pl.col("ticker").is_in(mismatch.implode()))
print(f"Dropped {len(mismatch):,} contracts whose last trade does not match Kalshi's reported last price (paper: 63)")

Dropped 18 contracts whose last trade does not match Kalshi's reported last price (paper: 63)


### Yes and No observations

Each traded Yes price `p` has a matching No price `1 - p` paid by the other side, so every snapshot gives two observations: one for buying Yes and one for buying No.

In [13]:
yes_rows = snapshots.with_columns(pl.lit("yes").alias("side"), pl.col("yes_price").alias("price"),
                                  (pl.col("result") == "yes").alias("won"))
no_rows = snapshots.with_columns(pl.lit("no").alias("side"), (1 - pl.col("yes_price")).round(4).alias("price"),
                                 (pl.col("result") == "no").alias("won"))
observations = pl.concat([yes_rows, no_rows]).drop("yes_price")

print(f"{snapshots['ticker'].n_unique():,} Yes contracts, {snapshots['event_ticker'].n_unique():,} events, "
      f"{len(snapshots):,} Yes prices, {len(observations):,} Yes and No prices")
print(f"Average daily observations per Yes contract: {len(snapshots) / snapshots['ticker'].n_unique():.2f} (paper: 3.5)")

41,721 Yes contracts, 11,678 events, 158,648 Yes prices, 317,296 Yes and No prices
Average daily observations per Yes contract: 3.80 (paper: 3.5)


### Check against the paper

**Headline counts.**

In [14]:
def headline_counts(snaps):
    return [snaps["event_ticker"].n_unique(), snaps["ticker"].n_unique(), len(snaps), 2 * len(snaps)]


pl.DataFrame({
    "count": ["events", "Yes contracts", "Yes prices", "Yes and No prices"],
    "ours": headline_counts(snapshots),
    "main_listing_only": headline_counts(snapshots.filter(~pl.col("ticker").is_in(extra_tickers.implode()))),
    "paper": [12_403, 46_282, 156_986, 313_972],
}).with_columns((pl.col("ours") / pl.col("paper")).round(2).alias("ratio"))

count,ours,main_listing_only,paper,ratio
str,i64,i64,i64,f64
"""events""",11678,9946,12403,0.94
"""Yes contracts""",41721,32837,46282,0.9
"""Yes prices""",158648,141185,156986,1.01
"""Yes and No prices""",317296,282370,313972,1.01


**Table 1: Yes contracts by how far back they have prices.** Each contract is counted by the furthest day before close with a price.

In [15]:
paper_table_1 = [19_338, 12_861, 1_723, 1_083, 1_311, 1_288, 1_461, 159, 150, 154, 6_754]

(snapshots
 .group_by("ticker").agg(pl.col("days_before").max().alias("furthest_day"))
 .group_by("furthest_day").len().sort("furthest_day")
 .with_columns(pl.Series("paper", paper_table_1)))

furthest_day,len,paper
i32,u32,i64
0,13203,19338
1,14116,12861
2,1652,1723
3,1065,1083
4,1207,1311
…,…,…
6,1424,1461
7,164,159
8,152,150


**Table 2: Yes and No observations by price range.** The paper's ranges are symmetric around 50c: a Yes at 41–49c is a No at 51–59c, and 50c is a No at 50c. So "41c-50c" holds 41–49c and "50c-59c" holds 50–59c, which is how the paper's numbers add up to 313,972.

In [16]:
PRICE_RANGES = ["1c-10c", "11c-20c", "21c-30c", "31c-40c", "41c-50c", "50c-59c", "60c-69c", "70c-79c", "80c-89c", "90c-99c"]
paper_table_2 = [106_209, 20_395, 12_558, 10_049, 7_199, 8_351, 10_049, 12_558, 20_395, 106_209]


def price_range(price):
    """The paper's 10c price ranges (Table 2)."""
    cents = (price * 100).round(2)
    return (pl.when(cents <= 10).then(pl.lit("1c-10c"))
            .when(cents <= 20).then(pl.lit("11c-20c"))
            .when(cents <= 30).then(pl.lit("21c-30c"))
            .when(cents <= 40).then(pl.lit("31c-40c"))
            .when(cents < 50).then(pl.lit("41c-50c"))
            .when(cents < 60).then(pl.lit("50c-59c"))
            .when(cents < 70).then(pl.lit("60c-69c"))
            .when(cents < 80).then(pl.lit("70c-79c"))
            .when(cents < 90).then(pl.lit("80c-89c"))
            .otherwise(pl.lit("90c-99c")))


observations = observations.with_columns(price_range(pl.col("price")).alias("price_range"))

table_2 = (
    observations.group_by("price_range").len()
    .join(pl.DataFrame({"price_range": PRICE_RANGES, "paper": paper_table_2, "order": range(10)}), on="price_range")
    .sort("order").drop("order")
    .with_columns((100 * pl.col("len") / pl.col("len").sum()).round(1).alias("percent"),
                  (100 * pl.col("paper") / pl.col("paper").sum()).round(1).alias("paper_percent"))
)
table_2

price_range,len,paper,percent,paper_percent
str,u32,i64,f64,f64
"""1c-10c""",104662,106209,33.0,33.8
"""11c-20c""",20583,20395,6.5,6.5
"""21c-30c""",13581,12558,4.3,4.0
"""31c-40c""",11005,10049,3.5,3.2
"""41c-50c""",8149,7199,2.6,2.3
"""50c-59c""",9485,8351,3.0,2.7
"""60c-69c""",11005,10049,3.5,3.2
"""70c-79c""",13581,12558,4.3,4.0
"""80c-89c""",20583,20395,6.5,6.5


## 5. Replication

*To do:* Return by 10c price bucket before and after fees, price regression, makers vs takers, by category and year.

## 6. Out of sample

*To do:* The same tables for May 2025 to October 2026.

## 7. Strategy backtest

*To do:* Weekly favourite buying, 85–97c, 1–21 days to close; NAV, performance and tail statistics.

## 8. Calibration model

*To do:* Logistic regression fit on pre-May-2025 data, reliability check, out-of-sample comparison with section 7.

## 9. Conclusions

*To do:* Findings and judgment.